# 1. Basic RAG 구축 실습

이 노트북에서는 LangChain을 사용하여 기본적인 RAG(Retrieval-Augmented Generation) 파이프라인을 구축합니다.
한국은행 업종 보고서를 데이터로 사용하여 질문에 답변하는 봇을 만듭니다.

**목표:**
1. PDF 문서를 로드하고 청크(Chunk)로 분할합니다.
2. Vector DB(Chroma)에 문서를 임베딩하여 저장합니다.
3. **Custom Tool**을 정의하여 Agent가 능동적으로 검색을 수행하도록 만듭니다.
4. 현재 구현의 **문제점**을 확인합니다.

이제 환경 변수를 등록해줍시다.

- 서비스를 활용하기 위해 API KEY 값이 필요합니다. 지정된 환경변수에 API KEY를 저장하면 쉽게 API를 연동할 수 있습니다.
- .env에 환경변수를 저장하고 한 번에 불러옵니다.
- .env 파일은 기본적으로 숨김파일이므로, colab에서 볼 때는 아래와 같이 눈 표시(파란색 원)를 체크해서 파일을 볼 수 있습니다.

In [1]:
import sys
import os
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv(override=True)

# PROJECT_ROOT를 .env에서 읽기 (없으면 현재 디렉토리)
project_root = os.getenv("PROJECT_ROOT", os.getcwd())

# 프로젝트 루트가 유효하지 않으면, 현재 위치에서 상위로 찾기
if not os.path.exists(os.path.join(project_root, "app")):
    # 상위 디렉토리 탐색
    current = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(current, "app")):
            project_root = current
            break
        current = os.path.dirname(current)

# Working Directory 설정
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import app

print(f"✅ Project Root: {project_root}")
print(f"✅ Working Directory: {os.getcwd()}")

✅ Project Root: /mnt/c/Users/hyoun/Desktop/github/llmops_class
✅ Working Directory: /mnt/c/Users/hyoun/Desktop/github/llmops_class


In [2]:
# API Key 확인 (선택 사항)
if "OPENAI_API_KEY" not in os.environ:
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")

## LangChain Review

## LangChain Chat LLM (Chat completion LLM) 사용하기

Chat LLM은 메시지 리스트를 입력 받아 응답을 생성하는 대화형 LLM입니다. 챗봇, 대화형 워크플로우, 대화형 RAG, 에이전트 개발 등에 적용되는 가장 보편적인 모델입니다. 데이터를 처리하려면 invoke 메서드에 메시지 목록을 전달하면 됩니다.

LangChain [Chat llm](https://python.langchain.com/docs/integrations/chat/) 객체는 다음과 같이 생성합니다.

In [3]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(model="gpt-5-mini", model_provider="openai", temperature=0.6)

/home/hukim/env_langchain_123/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


- model : 사용할 llm 모델 명칭
- model_provider : `openai`, `anthropic`, `google_genai` 등. pip install로 [제품사 패키지 추가설치 필요](https://python.langchain.com/docs/integrations/chat/#featured-providers).
    - 예를 들어 openai의 경우 langchain-openai 설치
- temperature : 낮을 수록 일관되고 높을 수록 무작위적이고 창의적이 됩니다.

.invoke 메서드로 LLM에 프롬프트 텍스트를 전달합니다. 결과는 AIMessage 타입의 데이터입니다.

In [4]:
ai_response = llm.invoke("안녕하세요? 당신은 어떤 모델인가요?")

(1) Response 내용 : 응답 텍스트는 content 속성에 있습니다.

In [5]:
ai_response.content

'안녕하세요! 저는 OpenAI가 만든 대형 언어 모델인 ChatGPT입니다(기본적으로 GPT-4 계열 기반). 텍스트를 이해하고 생성하는 능력이 있어 질문에 답하고, 글을 쓰고 고치고, 번역하고, 요약하고, 코드 작성·디버깅을 돕고, 학습·브레인스토밍·계획 수립 등을 도와드릴 수 있습니다.\n\n제한 사항도 있습니다:\n- 지식은 2024년 6월까지로 최신 정보나 실시간 웹 검색 결과는 반영되지 않습니다.\n- 때때로 부정확하거나 잘못된 정보를 만들 수 있으니 중요한 결정에는 전문가 확인을 권합니다(의료·법률·재무 등).\n- 인터넷에 직접 접속하거나 실제 행위를 대신 수행할 수는 없습니다.\n- 개인 민감 정보는 공유하지 않는 것이 안전합니다.\n\n원하시면 더 기술적·구체적인 설명(훈련 방법, API 사용 등)도 드릴 수 있어요. 무엇을 도와드릴까요?'

(2) 응답 관련 메타데이터도 확인할 수 있습니다.

llm provider의 api platform에서 지원 모델과 토큰 당 가격을 확인 가능 : [openai](https://platform.openai.com/docs/pricing)

In [6]:
ai_response.response_metadata

{'token_usage': {'completion_tokens': 619,
  'prompt_tokens': 16,
  'total_tokens': 635,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 384,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}},
 'model_provider': 'openai',
 'model_name': 'gpt-5-mini-2025-08-07',
 'system_fingerprint': None,
 'id': 'chatcmpl-DcBevBiPPrSGHFnh5Wep6xt8mokYw',
 'service_tier': 'default',
 'finish_reason': 'stop',
 'logprobs': None}

(3) langchain llm의 response 데이터 타입은 Message의 한 종류인 `AIMessage` 입니다.

In [7]:
type(ai_response)

langchain_core.messages.ai.AIMessage

In [8]:
ai_response

AIMessage(content='안녕하세요! 저는 OpenAI가 만든 대형 언어 모델인 ChatGPT입니다(기본적으로 GPT-4 계열 기반). 텍스트를 이해하고 생성하는 능력이 있어 질문에 답하고, 글을 쓰고 고치고, 번역하고, 요약하고, 코드 작성·디버깅을 돕고, 학습·브레인스토밍·계획 수립 등을 도와드릴 수 있습니다.\n\n제한 사항도 있습니다:\n- 지식은 2024년 6월까지로 최신 정보나 실시간 웹 검색 결과는 반영되지 않습니다.\n- 때때로 부정확하거나 잘못된 정보를 만들 수 있으니 중요한 결정에는 전문가 확인을 권합니다(의료·법률·재무 등).\n- 인터넷에 직접 접속하거나 실제 행위를 대신 수행할 수는 없습니다.\n- 개인 민감 정보는 공유하지 않는 것이 안전합니다.\n\n원하시면 더 기술적·구체적인 설명(훈련 방법, API 사용 등)도 드릴 수 있어요. 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 619, 'prompt_tokens': 16, 'total_tokens': 635, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DcBevBiPPrSGHFnh5Wep6xt8mokYw', 'service_tier': 'default', 'finish_reason': 'stop', 'logprob

### LLM과 메시지 리스트

우리가 사용하는 ChatLLM은 대화형 데이터를 처리하기 위해 SystemMessage, HumanMessage, AIMessage, ToolMessage 객체를 사용하여 [메시지](https://python.langchain.com/docs/concepts/messages/)를 정의합니다.

In [9]:
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage

messages = [
    SystemMessage("당신은 오직 한국어로 답변하는 친절한 AI입니다. 간결하게 답변합니다."),
    HumanMessage("Explain RAG(Retrieval-Augmented Generation.)"),
]

llm.invoke(messages)

AIMessage(content='RAG(Retrieval‑Augmented Generation)는 생성 모델(예: LLM)에 외부 지식(문서, 데이터베이스 등)을 실시간으로 검색해 제공함으로써 더 정확하고 최신의 답변을 만들게 하는 방법론입니다.\n\n핵심 아이디어\n- 생성 모델이 모든 사실을 내장할 필요 없이, 필요한 정보를 검색기(retriever)로 찾아와 그 정보를 바탕으로 텍스트를 생성(generator)함.\n- 검색된 문서(근거, context)를 조건으로 하여 모델이 응답을 생성하므로 근거 기반(grounded) 응답이 가능해짐.\n\n구성 요소\n1. Retriever: 질의와 문서의 유사도를 계산해 관련 문서(조각)를 뽑아냄.\n   - 전통적: BM25(키워드 기반)\n   - 현대적: Dense retriever(임베딩 + ANN: DPR, SBERT + FAISS/Milvus/Pinecone)\n2. Index/Vector DB: 문서 임베딩을 저장하고 빠르게 검색하는 인프라.\n3. Reranker(선택적): 후보 문서들 중 더 정확한 순서를 재평가.\n4. Generator: 검색 결과를 입력으로 받아 최종 답변 생성(엔드 투 엔드 LLM).\n5. Pipeline/Prompting: 검색 결과 결합(컨텍스트 구성, citation 포함) 및 프롬프트 설계.\n\n주요 아키텍처 스타일\n- Retrieve-then-Generate: 검색결과를 하나의 긴 컨텍스트로 합쳐서 생성(간단).\n- Fusion-in-Decoder (FiD): 여러 문서별로 인코딩 후 디코더에서 병합해서 생성(성능 좋음).\n- RAG-Sequence / RAG-Token: 문서별로 생성 확률을 통합하는 방식(논문 기반).\n\n장점\n- 최신/대규모 지식에 접근 가능(모델 재학습 불필요).\n- 모델의 파라미터를 작게 유지하면서도 사실성 향상.\n- 근거 제공(출처 표기)로 신뢰성 개선 가능.\n\n한계 및 위험\n- 검색 실패(관련 문서 누락) → 잘못

### LangChain 에이전트

LangGraph **Agent**는 주어진 작업을 수행하기 위해 연동된 외부 도구를 호출하여 문제를 해결합니다.

LLM(언어 모델)을 기반으로 하며, 그래프 내에서 **도구(Tool or Functions)**와 상호작용하여 LLM에게 어려운 작업을 처리할 수 있습니다. LLM은 자연어 처리 능력만으로 제한적일 수 있으므로, 특정 작업(예: 데이터 검색, 계산, 파일 읽기 등)을 수행하기 위해 외부의 도구를 호출합니다.


```
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from datetime import date

# 오늘 날짜를 YYYY-MM-DD 형식으로 가져옵니다.
today_date = date.today().strftime("%Y-%m-%d")

# Agent 생성 (User-requested style)

tools = file_management_tools
system_prompt = f"""당신은 친절하게 답변하는 대화형 에이전트입니다.
                    유저의 요청을 처리하기 위해 필요 시 Tool을 호출해 사용할 수 있습니다.
                    유저가 시점을 밝히지 않았다면 현재 시점 기준으로 검색하세요.
                    오늘의 날짜 : {today_date} """


test_agent = create_agent("openai:gpt-4o", tools=tools, system_prompt=system_prompt)

```

## RAG Agent 구축하기

### 1. 데이터 로드 및 전처리

이제 RAG 구축을 위한 데이터를 읽어옵니다.

- 한국은행에제 RAG 구축을 위한 데이터를 읽어옵니다.

- 한국은행 [주력산업 모니터링 보고서](https://www.bok.or.kr/portal/singl/newsData/list.do?pageIndex=&targetDepth=3&menuNo=201127&syncMenuChekKey=1&depthSubMain=&subMainAt=&searchCnd=1&searchKwd=&depth2=201042&depth3=201127&date=&sdate=&edate=&sort=1&pageUnit=10)입니다.

In [10]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import glob

# 데이터 경로 설정 (project_root 기반)
DATA_DIR = os.path.join(project_root, "data/bok_major_industry_reports")

# 모든 PDF 파일 경로 찾기
pdf_files = glob.glob(os.path.join(DATA_DIR, "*.pdf"))
print(f"Found {len(pdf_files)} PDF files: {pdf_files}")

docs = []
for file_path in pdf_files:
    loader = PyPDFLoader(file_path)
    docs.extend(loader.load())

print(f"Total pages loaded: {len(docs)}")

Found 4 PDF files: ['/mnt/c/Users/hyoun/Desktop/github/llmops_class/data/bok_major_industry_reports/2024_1사분기_주력산업_모니터링_보고서.pdf', '/mnt/c/Users/hyoun/Desktop/github/llmops_class/data/bok_major_industry_reports/2024_2사분기_주력산업_모니터링_보고서.pdf', '/mnt/c/Users/hyoun/Desktop/github/llmops_class/data/bok_major_industry_reports/2024_3사분기_주력산업_모니터링_보고서.pdf', '/mnt/c/Users/hyoun/Desktop/github/llmops_class/data/bok_major_industry_reports/2024_4사분기_주력산업_모니터링_보고서.pdf']
Total pages loaded: 154


In [11]:
# 문서 분할 (Chunking)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(docs)
print(f"Created {len(docs)} chunks.")

Created 274 chunks.


In [12]:
# 데이터 확인
from IPython.display import display, Markdown

print("--- Sample Chunks ---")
for doc in docs[10:15]:
    display(Markdown(doc.page_content))
    print("=====================================\n\n")

--- Sample Chunks ---


2024년 1/4분기 주력산업 모니터링 보고서(Ⅰ. 반도체)
- 3 -
향후 전망 1. 2024년 2/4분기 중 반도체 수출은 글로벌 AI 서비스 확산9)에 따른 고부가가치 메모리반도체 수요 확대10), 메모리반도체 가격 상승세 지속 등으로 상승흐름을 이어나갈 것으로 예상된다. 반도체생산은업체들이업황개선에따른이익극대화를위해가동률을점차정상화11)하는가운데,제품별재고수준에따라선별적으로생산수준을조정해나갈것으로보인다.아울러HBM증설등업체들의설비투자12)도확대될전망이다.2. 메모리반도체는 고부가가치 메모리반도체 수요 증가13)14)와 더불어 하반기 중 레거시 제품(PC15)16), 모바일17), 일반 서버18) 등)의 수요가 개선되면서 견조한 성장세를 보일 것으로 전망된다. 아울러 메모리반도체시장이초과수요구간에진입하면서메모리반도체가격상승세19)가하반기까지이어질것으로예상된다.재고율은 출하 증가 및 감산에 따라 재고율 하락메모리반도체의 초과수요로 가격 상승세 지속 예상[그림 1.4] 반도체 제조업 재고율1) [그림 1.5] 메모리반도체수급전망1)

주: 1) (재고지수/출하지수)×100,S.A.기준자료: 통계청   주: 1) 점선은 전망치 자료: Gartner(24.4월)9) AI 반도체 시장 전망(십억불, Gartner) :(2022)42.2→(2023)53.7→(2024e)71.3→(2025e) 92.0→(2026e)116.310) AI 플랫폼 성장으로 고객 맞춤형(용량, 성능, 특화 기능 등) HBM에 대한 요구가 증대되는 가운데, 업체들 간 수익성이 높은 HBM 공급을 확대하기 위한 경쟁이 심화되고 있다. SK하이닉스와 마이크론이 2024년 2/4분기 중 HBM3e(5세대) 8단 제품을, 삼성전자는 동 분기중 HBM3e(5세대) 8단 및 12단 제품을 납품할 계획이다(삼성전자 실적발표, 4.30일). 11) 업계에 따르면, D램의 경우 총 웨이퍼 투입량이 2024년 1/4분기를 저점으로 상승세가 확대될 것으로 전망하고 있다.12) SK하이닉스는 HBM 생산능력을 확장하기 위해 낸드플래시 생산공장으로 계획하였던 청주 반도체 공장(M15X)을 차세대 D램 생산시설로 변경하여 건설한다는 계획을 발표(4.24일)하였다. SK하이닉스는 2025년 11월 양산을 목표로 공장 건설에 5조 3,000억원을 투자하고, 장기적으로는 총 20조원 이상의 투자를 집행할 계획이다.13) AI 서비스가 진화하는 과정에서 저장 공간 수요가 지속적으로 확대될 것으로 전망하고 있다. 오픈 AI가 최근 공개한 영상 기반 AI 모델 ‘소라(Sora)‘의 경우 기존 텍스트 기반 거대언어모델(LLM)에 비해 훨씬 더 큰 저장 공간이 필요하다. 14) 빅테크 업체들이 자체 AI 칩 개발에 투자를 확대하는 가운데, 반도체 업계도 AI 칩 개발 경쟁에 뛰어들고 있다. 구글이 자체 AI 전용칩인 TPU v5p를 출시한 가운데 메타도 맞춤형 AI 칩인 MTIA v2을 공개(4.10일)하였다. 또한 인텔은 ‘가우디3’ 칩을 공개하고 2024년 3/4분기 중 델(Dell), HP, 슈퍼마이크로 등 PC 및 서버 업체에 납품할 계획임을

AI 칩 개발에 투자를 확대하는 가운데, 반도체 업계도 AI 칩 개발 경쟁에 뛰어들고 있다. 구글이 자체 AI 전용칩인 TPU v5p를 출시한 가운데 메타도 맞춤형 AI 칩인 MTIA v2을 공개(4.10일)하였다. 또한 인텔은 ‘가우디3’ 칩을 공개하고 2024년 3/4분기 중 델(Dell), HP, 슈퍼마이크로 등 PC 및 서버 업체에 납품할 계획임을 발표(인텔 비전 2024, 4.9일)하였다. 15) 팬데믹으로 지연되었던 PC 구매 교체수요 주기 도래, Window 10 서비스 종료(2025.10.14일) 및 AI PC 출시에 따른 기존 PC 교체 및 메모리반도체 채용량 증대를 전망하고 있다.16) AI PC는 AI 작업을 최적화·가속화하도록 설계된 전용 AI 가속기나 신경망처리장치(NPU) 등이 장착된 PC이며, 클라우드(외부 서버) 연결 없이 개인용 PC에 생성형 AI를 탑재하여 보안에 유리하다. 인텔에 이어 AMD, 퀄컴, 엔비디아 및 애플도 차세대 칩 개발을 통해 AI PC 신제품을 출시할 예정이다.17) 모바일 OEM 업체들의 상반기 중 적극적인 도매매출(Sell-in) 영향으로 유통재고가 다소 상승하여 하반기 중 반도체 수요증가는 제한적일 것으로 보인다. 18) 생성형 AI 기술의 고도화, AI 기술의 학습(training)에서 추론(inferencing)으로 확산 등 AI 관련 수요 강세가 지속되는 가운데, 2017년~2018년 설치된 클라우드 데이터센터의 서버(일반 서버) 교체 수요도 점차 발생할 것으로 보인다.19) 메모리반도체 가격 전망(전년동기대비, Trendforce)    D램 고정가격 등락률 : (2024.1/4)+15.0%→(2/4)+11.8%→(3/4)+11.2%→(4/4) +5.0%    낸드 고정가격 등락률 :(2024.1/4)+18.0%→(2/4)+6.4%→(3/4)+7.5%→(4/4) +0.7%

2024년 1/4분기 주력산업 모니터링 보고서(Ⅰ. 반도체)
- 4 -
3. 파운드리도 지난해의 부진20)에서 벗어나 AI 수요 증가 등에 힘입어 업황이 점차 개선21)될 전망이다. Pure-Play파운드리기업과종합반도체기업(IDM)들간경쟁이심화22)되는가운데,선단공정,첨단패키징23)부문등선행기술우위에기반한Pure-play업체의성장세가IDM을상회하면서시장점유율이증가24)할것으로예상된다.4. 한편 미국 상무부는 작년 12월 반도체지원법(CHIPS Act)에 따른 보조금 지원대상을 발표하면서 인텔, TSMC에 이어 삼성전자에 대해서도 64억 달러 규모의 보조금을 지급할 계획25)을 발표(4.15일)하였다. 이러한미국상무부의보조금지원조치는반도체공급망의자급체제를구축하고중국과의첨단반도체생산기술격차를확대하는데기여할것으로보인다.반도체기술패권을차지하기위한각국의지원노력26)27)이가속화되는가운데,국내기업들이반도체기술주도권을확보하기위해서는세제혜택이외에도주요인프라(도로망·전기·용수등)지원,고학력자수급및보조금지급등정부의적극적인지원노력이필요할것으로판단된다.2024~2028년 중 파운드리 시장은 연평균 10.4% 성장 예상 주요 반도체 기업의 미국 투자규모는 약 4,500억 달러로 약 56,000여개의 일자리 창출을 기대[그림 1.6] 전세계 파운드리 시장 전망[표 1.2] 주요 반도체 기업의 미국 투자계획

(단위 : 억달러)기업명종류위치투자규모인 텔반도체애리조나 320반도체오하이오  280패키징뉴멕시코75R&D오리건360TSM C반도체애리조나  650삼 성 전 자반도체텍사스  450마 이 크 론반도체아이다호250반도체뉴욕주1,000SK 하 이 닉 스패키징인디애나40자료: Techinsigts자료: SIA(미국 반도체산업협회)20) 세계 파운드리 시장은 2023년 중 1,234억 달러를 기록하여 전년(1,421억 달러) 대비 13% 감소하였으며, 이는 2001년 이후 최대 매출 감소치이다.21) TSMC는 2024년 세계 파운드리 시장 성장률을 AI 분야를 제외한 스마트폰, PC 교체 수요 정체 및 최근 전기차 캐즘(대중화 직전 수요 둔화) 등 자동차 반도체 수주량 감소로 당초 예상치 20%에서 10% 중후반으로 하향 조정하였으나, AI 관련 매출은 매년 50% 씩 성장하고 있다고 밝혔다.(TSMC 실적발표, 4.18일) 삼성전자는 파운드리 사업부의 매출이 1/4분기 저점을 찍고 2/4분기부터 반등하여 전기대비 두자리 수 매출 성장을 기대한다고 발표하였다.(삼성전자 실적발표, 4.30일)22) Pure-Play Foundry는 제조를 위탁받아 반도체를 생산하는 회사로 TSMC(대만), UMC(대만), GlobalFoundry(미국), SMIC(중국) 등이 있다. IDM(Integrated Device Manufacturer)은 위탁생산과 동시에 자체 IC, 기타 반도체를 제조 및 판매하는 종합반도체 회사로 삼성전자, 인텔 등이 해당된다.23) TSMC의 “CoWoS(Chip on Wafer on Substrate)”는 중간기판(인터포저 판) 위에 메모리와 로직 반도체를 올리는 기술로 기존 패키징 보다 면적이 줄고, 칩 간 연결을 빠르게 하여 고성능컴퓨팅(HPC), AI, 데이터 센터 등 다양한 응용처에서 광범위하게 사용되고 있다.24) Pure-Play 파운드리 기업이 2023년 중 파운드리 전체 시장의 약 84% 이상을 차지하였고, 2028년 중 87% 이상을

### 2. Vector DB 구축 (Chroma)

임베딩 모델로는 `text-embedding-3-large`를 사용하고, Vector DB는 `Chroma`를 사용합니다.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from IPython.display import display, Markdown

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large"
)

# 벡터 저장소 생성 (메모리 내)
report_vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory=f"{project_root}/data/chroma_db_notebooks", 
    # 테스트를 위해 DB를 저장하는 대신 Inmemory로 생성합니다. 
    # 이미 해당 경로에 vectorsotre가 구축되어있습니다.        
    collection_name="basic_rag"
)

report_retriever = report_vectorstore.as_retriever()

# 검색 테스트
query = "2024년 4분기 반도체 업황"
retrieved_docs = report_retriever.invoke(query)

print(f"-- Search Results for '{query}' --")
for doc in retrieved_docs:
    display(Markdown(str(doc.metadata)))
    print(doc.page_content[:200] + "...")
    print("=====================================\n\n")

### 3. Agent Tool (Custom Tool Definition)

여기서는 `create_retriever_tool` 대신 **`@tool` 데코레이터**를 사용하여 커스텀 툴을 정의합니다.
이렇게 하면:
1. `limit` 등을 인자로 받아 LLM이 검색 개수를 조절할 수 있습니다.
2. 검색 결과 포맷팅(Metadata 포함)이나 로깅을 우리가 원하는 대로 제어할 수 있습니다.

In [ ]:
from langchain_core.tools import tool

@tool(parse_docstring=True)
def search_bok_reports(query: str, limit: int = 4) -> str:
    """Search the Bank of Korea industry reports for relevant information.

    Args:
        query: The search query to find relevant information in the reports.
        limit: The maximum number of documents to retrieve. Defaults to 4.
    """
    print(f"[Tool Log] Searching for '{query}' with limit={limit}...")

    # Retriever의 'k' 값을 동적으로 조절하기 위해 search_kwargs 업데이트
    report_retriever = report_vectorstore.as_retriever()
    docs = report_retriever.invoke(query, k=limit)

    result_text = ""
    for i, doc in enumerate(docs):
        # Metadata 정보도 답변 생성에 참고할 수 있도록 포함
        meta_str = str(doc.metadata)
        result_text += f"[Document {i+1}]\nMetadata: {meta_str}\nContent: {doc.page_content}\n\n"

    return result_text

tools = [search_bok_reports]

In [ ]:
# Custom Tool 테스트 (Direct Invocation)
# 에이전트 연결 전, 도구가 잘 작동하는지 단독으로 테스트합니다.
test_query = "반도체 수출 전망"
print(f"Testing tool with query: {test_query}\n")

tool_output = search_bok_reports.invoke({"query": test_query, "limit": 2})
print(tool_output)

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

# Agent 생성 (User-requested style)
# Note: 'create_agent' signature depends on the specific package version/wrapper user has.
rag_agent = create_agent("openai:gpt-4o", tools=tools)

# Agent 테스트
input_query = "24년도 2분기 반도체 업황을 핵심 위주로 요약해줘."

response = rag_agent.invoke({"messages": [("user", input_query)]})

print(f"Q: {input_query}\n")
print(f"A: {response['messages'][-1].content}")

### 4. Self-Querying Retriever (셀프 쿼리 리트리버)로 성능 개선하기

Self-Querying Retriever는 사용자의 **자연어 질문을 LLM이 직접 분석**하여, 검색에 필요한 **의미 검색용 쿼리 문자열**과 **메타데이터 필터 조건**을 **자동으로 생성**하고 이를 사용하여 벡터 저장소를 검색하는 지능적인 RAG 기법입니다. 사용자가 복잡한 필터 구문을 알 필요 없이, 일상적인 대화처럼 질문해도 정교한 필터링이 가능해집니다.

In [ ]:
import re, os
from app.utils.pymupdf4llm_loader import PyMuPDF4LLMLoader
import glob

DATA_DIR = os.path.join(project_root, "data/bok_major_industry_reports")
pdf_files = glob.glob(os.path.join(DATA_DIR, "*.pdf"))

docs = []
for file_path in pdf_files:
    filename = os.path.basename(file_path)
    # 예: "2024_1사분기_..."
    match = re.search(r"(\d{4})_(\d)사분기", filename ) # 메타데이터 추출
    year = int(match.group(1)) if match else 2024
    quarter = int(match.group(2)) if match else 0
    print(f"Loading {filename} (Year={year}, Quarter={quarter})...")
    
    loader = PyMuPDF4LLMLoader(file_path, mode="page", extract_images=False)
    file_docs = loader.load()

    for doc in file_docs:
        doc.metadata["year"] = year
        doc.metadata["quarter"] = quarter
        doc.metadata["source"] = filename

    docs.extend(file_docs)

In [ ]:
from langchain_text_splitters import MarkdownTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.schema import AttributeInfo

# 1. 텍스트 분할 (마크다운 헤더 구조 고려)
text_splitter = MarkdownTextSplitter(chunk_size=2000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)



# 2. 벡터 저장소 생성
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=OpenAIEmbeddings(model="text-embedding-3-large"),
    # persist_directory=f"{project_root}/data/chroma_db", 
    # 테스트를 위해 DB를 저장하는 대신 Inmemory로 생성합니다. 
    # 이미 해당 경로에 vectorsotre가 구축되어있습니다.        
    collection_name="self_query"
)
print("Vectorstore 구축 완료!")


# 3. 메타데이터 필드 정보
metadata_field_info = [
    AttributeInfo(
        name="year",
        description="해당 보고서의 연도를 나타내는 정수 값 (예: 2024)",
        type="integer",
    ),
    AttributeInfo(
        name="quarter",
        description="해당 보고서의 분기를 나타내는 숫자로 1, 2, 3, 4 중 하나.",
        type="integer",
    ),
]
document_content_description = "Bank of Korea Industry Reports"

# 4. SelfQueryRetriever 생성
llm = ChatOpenAI(model="gpt-4o", temperature=0).with_config({"tags": ["exclude_from_stream"]})

sq_retriever = SelfQueryRetriever.from_llm(
    llm,
    vectorstore,
    document_content_description,
    metadata_field_info,
    verbose=False
)

In [ ]:
from IPython.display import display, Markdown

print("24년도 2분기 조선업 업황")
docs = sq_retriever.invoke("24년도 2분기 조선업 업황")
for doc in docs:
    display(Markdown(str(f"#{doc.metadata['year']}년, {doc.metadata['quarter']}분기")))
    print("=====================================")
print("24년도 4분기 반도체 산업 동향")
docs = sq_retriever.invoke("24년도 4분기 반도체 산업 동향")
for doc in docs:
    display(Markdown(str(f"#{doc.metadata['year']}년, {doc.metadata['quarter']}분기")))
    print("=====================================")
print("24년도 2분기 이후의 핸드폰 관련 자료")
docs = sq_retriever.invoke("24년도 2분기 이후의 핸드폰 관련 자료")
for doc in docs:
    display(Markdown(str(f"#{doc.metadata['year']}년, {doc.metadata['quarter']}분기")))
    print("=====================================")


### 5. Multimodal Retriever (멀티모달 리트리버)
이제 Self-Querying Retriever에 추가적으로 이미지와 텍스트를 동시에 검색할 수 있는 리트리버를 만들어보겠습니다.
> 💡 **Self-Query vs Multimodal — 코드 차이점**
> 
> 아래 데이터 로드 및 VectorStore 구축 코드는 Self-Query 섹션(섹션 4)과 구조가 거의 동일합니다. 
> **핵심 차이점**은 `PyMuPDF4LLMLoader`의 설정에 있습니다:
> - `extract_images=True` → PDF 내 그래프/도표 이미지를 함께 추출합니다.
> - `image_output_dir` → 추출된 이미지가 저장될 경로를 지정합니다.
> - `model=ChatGoogleGenerativeAI(...)` → 추출된 이미지를 설명(caption)하기 위한 멀티모달 LLM을 연결합니다.
> 
> 이렇게 하면 텍스트뿐 아니라 **이미지 설명(caption)이 문서 청크에 포함**되어, 검색 시 그래프나 도표 관련 질문에도 관련 문서가 검색됩니다.

In [ ]:
import re, os
from app.utils.pymupdf4llm_loader import PyMuPDF4LLMLoader
from langchain_google_genai import ChatGoogleGenerativeAI
import glob

DATA_DIR = os.path.join(project_root, "data/bok_major_industry_reports")
pdf_files = glob.glob(os.path.join(DATA_DIR, "*.pdf"))


docs = []
for file_path in pdf_files:
    filename = os.path.basename(file_path)
    # 예: "2024_1사분기_..."
    match = re.search(r"(\d{4})_(\d)사분기", filename ) # 메타데이터 추출
    year = int(match.group(1)) if match else 2024
    quarter = int(match.group(2)) if match else 0
    print(f"Loading {filename} (Year={year}, Quarter={quarter})...")
    
    loader = PyMuPDF4LLMLoader(file_path, mode="page", 
                                extract_images=True, 
                                image_output_dir="./extracted_images",
                                model=ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0))
    file_docs = loader.load()

    for doc in file_docs:
        doc.metadata["year"] = year
        doc.metadata["quarter"] = quarter
        doc.metadata["source"] = filename

    docs.extend(file_docs)

In [ ]:
from langchain_text_splitters import MarkdownTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.schema import AttributeInfo

# 1. 텍스트 분할 (마크다운 헤더 구조 고려)
text_splitter = MarkdownTextSplitter(chunk_size=2000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# 2. 벡터 저장소 생성
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=OpenAIEmbeddings(model="text-embedding-3-large"),
    # persist_directory=f"{project_root}/data/chroma_db", 
    # 테스트를 위해 DB를 저장하는 대신 Inmemory로 생성합니다. 
    # 이미 해당 경로에 vectorsotre가 구축되어있습니다.        
    collection_name="multimodal"
)
print("Vectorstore 구축 완료!")

metadata_field_info = [
    AttributeInfo(
        name="year",
        description="해당 보고서의 연도를 나타내는 정수 값 (예: 2024)",
        type="integer",
    ),
    AttributeInfo(
        name="quarter",
        description="해당 보고서의 분기를 나타내는 숫자로 1, 2, 3, 4 중 하나.",
        type="integer",
    ),
]
document_content_description = "Bank of Korea Industry Reports"

llm = ChatOpenAI(model="gpt-4o", temperature=0).with_config({"tags": ["exclude_from_stream"]})

multimodal_retriever = SelfQueryRetriever.from_llm(
    llm,
    vectorstore,
    document_content_description,
    metadata_field_info,
    verbose=False
)

### 멀티모달 리트리버 테스트
이제 이미지가 포함된 문서를 검색하고, 검색 결과에서 이미지 경로를 추출하여 LLM이 활용할 수 있는 형태로 가공하는 과정을 확인합니다.
아래 셀에서는:
1. `multimodal_retriever`로 **그래프/도표 관련 질문**을 검색합니다.
2. 검색된 문서에서 **마크다운 이미지 링크**(`![alt](path)`)를 정규식으로 추출합니다.
3. 추출된 이미지 경로와 텍스트를 합쳐, 멀티모달 LLM에 전달할 수 있는 딕셔너리(`context`, `images`, `source`)로 정리합니다.

In [ ]:
retrieved_docs = multimodal_retriever.invoke("24년도 1분기 반도체 수출의 증가폭 관련 그래프를 설명해줘")

In [ ]:
retrieved_docs

In [ ]:
from pathlib import Path

BASE_DIR = Path(project_root) # notebook이 보고 있는 base dir입니다.
DATA_DIR = BASE_DIR / "data" / "extracted_images" # 이미지가 저장된 dir입니다.

combined_text = []
image_paths = []

# 마크다운 이미지 링크 패턴: ![alt](path)
img_pattern = re.compile(r'!\[.*?\]\((.*?)\)')

for image_doc in retrieved_docs:
    content = image_doc.page_content

    # A. 이미지 경로 추출
    found_paths = img_pattern.findall(content)

    for path in found_paths:

        filename = os.path.basename(path) # 파일명 추출
        abs_path = DATA_DIR / filename # 절대 경로 생성 (Agent가 파일을 열 때 사용)

        # 중복 제거 및 실제 파일 존재 여부 확인 (안전장치)
        if path not in image_paths:
            if abs_path.exists():
                image_paths.append(str(abs_path))

    # B. 텍스트 누적 (페이지 정보 포함하여 LLM이 출처를 알 수 있게 함)
    page_num = image_doc.metadata.get('page', 'Unknown')
    combined_text.append(f"--- [Page {page_num}] ---\n{content}")


In [ ]:
found_paths

In [ ]:
{
    "context": "\n\n".join(combined_text), # 검색된 텍스트 전체
    "images": image_paths,                 # 추출된 이미지 경로 리스트
    "source": "한국은행 8대 업종 모니터링 보고서" # 출처 태그
}

## 정리(Summary)

이 노트북에서는 RAG Agent를 **세 가지 방식**으로 점진적으로 발전시켜 보았습니다.

| 단계 | 방식 | 특징 | 한계 |
|------|------|------|------|
| 1 | **Basic RAG** | 단순 유사도 검색으로 관련 문서 반환 | 시점/분기 등 조건 필터링 불가 |
| 2 | **Self-Query RAG** | LLM이 메타데이터(연도, 분기)를 자동 추출 → 정교한 필터링 | 텍스트 정보만 활용, 도표/그래프 분석 불가 |
| 3 | **Multimodal RAG** | 텍스트 + 이미지를 동시에 처리 → 도표/그래프 분석 가능 | 이미지 캡셔닝 비용 및 시간 증가 |

**핵심:**
- **Custom Tool**(`@tool` 데코레이터)을 사용하면 검색 로직을 세밀하게 제어할 수 있습니다.
- **Self-Query Retriever**는 사용자가 별도의 필터 구문 없이도 자연어만으로 메타데이터 기반 검색이 가능하게 합니다.
- **멀티모달 처리**를 통해 보고서 내 그래프/도표까지 검색 대상에 포함시킬 수 있습니다.

> 👉 **다음 노트북**에서는 이 에이전트들의 품질을 **정량적으로 평가**(LLM-as-Judge)하고, **LangSmith**를 활용한 Observability 실습을 진행합니다.
